In [1]:
from multiprocessing import Pool
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile
import copy
from datetime import datetime
import pandas as pd
from collections import Counter
import numpy as np
import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from joblib import load
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2

In [13]:
output_dir = source_path + '/outputs/fine_tuning/'
script_name = source_path+"/scripts/fine-tuning.py"
input_preprocessed='icdar_train_df_patches_20250515_164130.csv'
selected_model = 'resnet18' #'DeiT-Tiny'  # Example model, can be changed
selected_classifier = 'logreg'
list_of_metrics = ['majority_vote', 'weighted_vote', 'most_probable']
is_progressive = False  # Set to True for progressive training

# launch fine-tuning

In [18]:
#parameters
args = DotDict(
    N_max=282,
    patches=True,
    input_filename=input_preprocessed,
    huggingface=False,
    pooling=False,  # if true in transformer models use pooling, if false only the cls token
    custom_transform=False,
    transform_mode='resize',
    save_h5=False,
    selected_model=selected_model,  # googlenet, alexnet
    truncation='remove head',
    running='new-laptop',
    saved='old-laptop',
    model_mode='truncated',  # 'truncation
    batch_size=64,
    select_cls=False,
    num_workers=4,
    pin_memory=True,
    show_image=False,
    is_progressive=is_progressive,  # True for progressive training
    checkpoint_path = output_dir+"checkpoint.pt",
    save_path = output_dir,
    base_lr = 1e-4, #3e-5,
    total_epochs = 100,
    #weight_decay=1e-6,
    temperature=0.5,
    log_grad_norm = True,
    use_profiler = True,
    run_epochs = 8,
    plot_every = 5,
    patience = 20,
    warmup_epochs = 10,
    use_amp = False ,#mixed precision training,
    val_percentage= 1.0 ,#percentage of validation data used for linear evaluation,
    n_splits = 4,
    loss_criterion = 'CrossEntropyLoss',
    scheduler_name = 'CosineAnnealingWarmRestarts',  # e.g., 'CosineAnnealingLR', 'StepLR', etc.
    selected_classifier=selected_classifier,  # 'logreg', 'svm', 'rf', 'gbc', 'mlp', 'dt',
    optim_name='AdamW',  # e.g., 'Adam', 'SGD', 'AdamW'
    #script_mode='standalone'
)

In [21]:
run_experiment_threaded(args,script_name)  # Test a single run first

Starting experiment: file=icdar_train_df_patches_20250515_164130.csv, model=resnet18
[STDOUT] Output shape:  torch.Size([1, 512])
[STDOUT] Device is:  cuda
[STDOUT] val writers are:  {np.int64(256), np.int64(259), np.int64(4), np.int64(136), np.int64(137), np.int64(138), np.int64(139), np.int64(13), np.int64(141), np.int64(14), np.int64(16), np.int64(17), np.int64(143), np.int64(275), np.int64(274), np.int64(149), np.int64(23), np.int64(280), np.int64(151), np.int64(24), np.int64(29), np.int64(162), np.int64(34), np.int64(36), np.int64(37), np.int64(167), np.int64(41), np.int64(45), np.int64(173), np.int64(175), np.int64(48), np.int64(177), np.int64(50), np.int64(52), np.int64(53), np.int64(182), np.int64(184), np.int64(58), np.int64(186), np.int64(187), np.int64(190), np.int64(64), np.int64(194), np.int64(195), np.int64(72), np.int64(203), np.int64(206), np.int64(80), np.int64(82), np.int64(84), np.int64(215), np.int64(88), np.int64(217), np.int64(99), np.int64(102), np.int64(230), np

# functions

## reload

In [4]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod = reload_modules()

## scripts

In [10]:
def run_experiment_threaded(try_args, script_name):
    def stream_output(pipe, name):
        for line in iter(pipe.readline, ''):
            if line:
                print(f"[{name}] {line}", end='')

    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.yaml') as tmp:
        yaml.dump(try_args.__dict__, tmp)
        tmp_path = tmp.name

    print(f"Starting experiment: file={try_args.input_filename}, model={try_args.selected_model}")

    process = subprocess.Popen(
        ['python', script_name, '--config', tmp_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    # Start threads for live output
    stdout_thread = threading.Thread(target=stream_output, args=(process.stdout, 'STDOUT'))
    stderr_thread = threading.Thread(target=stream_output, args=(process.stderr, 'STDERR'))
    stdout_thread.start()
    stderr_thread.start()

    process.wait()
    stdout_thread.join()
    stderr_thread.join()

    print(f"Experiment finished with return code: {process.returncode}")
    return
class DotDict:
    def __init__(self, **entries):
        self.__dict__.update(entries)

    def __setitem__(self, key, value):
        setattr(self, key, value)

    def __getitem__(self, key):
        return getattr(self, key)

    def __repr__(self):
        return f"{self.__dict__}"
def load_config(path):
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        return DotDict(**config)